In [ ]:
data_dir = ""
scan_number_lcp = 1
scan_number_rcp = 2
input_dir = ""
output_dir = ""
crop = 50
align_x1 = 200
align_x2 = 450
align_y1 = 220
align_y2 = 450
ramp_x1 = 530
ramp_x2 = 600
ramp_y1 = 300
ramp_y2 = 400
norm_x1 = 520
norm_x2 = 620
norm_y1 = 200
norm_y2 = 300

## Define Imports

In [ ]:
import h5py
import glob
import numpy as np
import matplotlib.pyplot as plt
from skimage.segmentation import chan_vese
from pathlib import Path

from imaging_toolbox.ptychography import remove_ramp_and_unwrap_phase
from imaging_toolbox.alignment import align_data, fft_shift_data
from imaging_toolbox.utils import normalise_data

In [ ]:
def segmentation(
        data,
        y_range=(0, -1),
        x_range=(0, -1),
        circle=True,
        circle_y_range=(75, 100),
        circle_x_range=(25, 50)
        ):
    """ 
    Function to segment an image based on the chan vese algorithm.
    Can only be applied on arrays with real values
    """
    labels = chan_vese(
        data[y_range[0]:y_range[1],
             x_range[0]:x_range[1]],
             mu=0.05,
             init_level_set='checkerboard',
             dt=0.5
             )
    if circle==True:
        if np.average(
            labels[circle_y_range[0]:circle_y_range[1],
                   circle_x_range[0]:circle_x_range[1]]
            ) == 1:
            labels = 1-labels
    return labels

## Check Input Paths

In [ ]:
inpath = data_dir + input_dir

lcp_files = glob.glob(input_dir + f"/scan_{scan_number_lcp}/*.ptyr")
if len(lcp_files) == 0:
    raise FileNotFoundError(f"No ptychography files found in {inpath}/scan_{scan_number_lcp}")
lcp_fp = lcp_files[0]

rcp_files = glob.glob(input_dir + f"/scan_{scan_number_rcp}/*.ptyr")
if len(rcp_files) == 0:
    raise FileNotFoundError(f"No ptychography files found in {inpath}/scan_{scan_number_rcp}")
rcp_fp = rcp_files[0]

## Load LCP Data

In [ ]:
with h5py.File(lcp_fp, "r") as f:
    lcp_data = np.array(f["content/obj/Sscan_00G00/data"][0,crop:-crop,crop:-crop])
    lcp_amp = np.abs(lcp_data)
    lcp_phase = np.angle(lcp_data)

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(8, 4))

ax1.imshow(lcp_amp, cmap="bone")
ax1.set_title("Amplitude")
ax2.imshow(lcp_phase, cmap="bone")
ax2.set_title("Phase")

plt.show()

## Region Chosen for Alignment

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(8, 4))

ax1.imshow(lcp_amp, cmap="bone")
ax1.set_title("Alignment ROI")
ax1.axvline(align_x1,color='r')
ax1.axvline(align_x2,color='r')
ax1.axhline(align_y1,color='b')
ax1.axhline(align_y2,color='b')

ax2.imshow(lcp_amp[align_y1:align_y2,align_x1:align_x2], cmap="bone")
ax2.set_title("Alignment ROI Zoomed")

plt.show()

## Region Chosen for Phase Ramp Removal

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(8, 4))

ax1.imshow(lcp_phase, cmap="bone", vmin=-np.pi, vmax=np.pi)
ax1.set_title("Phase Ramp ROI")
ax1.axis("off")
ax1.axvline(ramp_x1,color='r')
ax1.axvline(ramp_x2,color='r')
ax1.axhline(ramp_y1,color='b')
ax1.axhline(ramp_y2,color='b')

rm_mask = np.zeros_like(lcp_amp, dtype="bool")
rm_mask[ramp_y1:ramp_y2, ramp_x1:ramp_x2] = True
ramp_corr_lcp_phase = remove_ramp_and_unwrap_phase(lcp_phase, mask=rm_mask)

ax2.imshow(ramp_corr_lcp_phase, cmap="bone", vmin=-3*np.pi, vmax=3*np.pi)
ax2.set_title("Corrected")
ax2.axis("off")

plt.show()

## Region Chosen for Normalisation

In [ ]:
fig,(ax1,ax2,ax3) = plt.subplots(nrows=1, ncols=3, figsize=(12, 4))

ax1.imshow(ramp_corr_lcp_phase.real, cmap='bone', vmin=-1.5*np.pi,vmax=1.5*np.pi)
ax1.axvline(norm_x1,color='r')
ax1.axvline(norm_x2,color='r')
ax1.axhline(norm_y1,color='b')
ax1.axhline(norm_y2,color='b')
ax1.set_title('Normalisation ROI')
ax1.axis("off")

norm_lcp_amp = normalise_data(
  lcp_amp,
  y_range=(norm_y1,norm_y2),
  x_range=(norm_x1,norm_x2),
  dtype="amplitude"
)

ax2.imshow(norm_lcp_amp, cmap="bone", vmin=0.2, vmax=1)
ax2.set_title("Normalised LCP Amp")
ax2.axis("off")

norm_lcp_phase = normalise_data(
  ramp_corr_lcp_phase.real,
  y_range=(norm_y1,norm_y2),
  x_range=(norm_x1,norm_x2),
  dtype="phase"
)

ax3.imshow(norm_lcp_phase, cmap="bone", vmin=-1, vmax=1)
ax3.set_title("Normalised LCP Phase")
ax3.axis("off")

plt.show()

## Load RCP Data

In [ ]:
with h5py.File(rcp_fp, "r") as f:
    rcp_data = np.array(f["content/obj/Sscan_00G00/data"][0,crop:-crop,crop:-crop])
    rcp_amp = np.abs(rcp_data)
    rcp_phase = np.angle(rcp_data)

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(8, 4))

ax1.imshow(rcp_amp, cmap="bone")
ax1.set_title("Amplitude")
ax2.imshow(rcp_phase, cmap="bone")
ax2.set_title("Phase")

plt.show()

## RCP Phase Ramp Removal

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(8,4))

ax1.imshow(rcp_phase, cmap="bone", vmin=-np.pi, vmax=np.pi)
ax1.set_title("Phase Ramp ROI")
ax1.axis("off")
ax1.axvline(ramp_x1,color='r')
ax1.axvline(ramp_x2,color='r')
ax1.axhline(ramp_y1,color='b')
ax1.axhline(ramp_y2,color='b')

rm_mask = np.zeros_like(rcp_amp, dtype="bool")
rm_mask[ramp_y1:ramp_y2, ramp_x1:ramp_x2] = True
ramp_corr_rcp_phase = remove_ramp_and_unwrap_phase(rcp_phase, mask=rm_mask)

ax2.imshow(ramp_corr_rcp_phase, cmap="bone", vmin=-3*np.pi, vmax=3*np.pi)
ax2.set_title("Corrected")
ax2.axis("off")

plt.show()

## RCP Normalisation

In [ ]:
fig,(ax1,ax2,ax3) = plt.subplots(nrows=1, ncols=3, figsize=(12,4))

ax1.imshow(ramp_corr_rcp_phase.real, cmap='bone', vmin=-1.5*np.pi,vmax=1.5*np.pi)
ax1.axvline(norm_x1,color='r')
ax1.axvline(norm_x2,color='r')
ax1.axhline(norm_y1,color='b')
ax1.axhline(norm_y2,color='b')
ax1.set_title('Normalisation ROI')

norm_rcp_amp = normalise_data(
  rcp_amp,
  y_range=(norm_y1,norm_y2),
  x_range=(norm_x1,norm_x2),
  dtype="amplitude"
)

ax2.imshow(norm_rcp_amp, cmap="bone", vmin=0.2, vmax=1)
ax2.set_title("Normalised RCP Amp")

norm_rcp_phase = normalise_data(
  ramp_corr_rcp_phase.real,
  y_range=(norm_y1,norm_y2),
  x_range=(norm_x1,norm_x2),
  dtype="phase"
)

ax3.imshow(norm_rcp_phase, cmap="bone", vmin=-1, vmax=1)
ax3.set_title("Normalised RCP Phase")

plt.show()

## Alignment

In [ ]:
seg_lcp_amp = segmentation(norm_lcp_amp[align_y1:align_y2, align_x1:align_x2])
seg_rcp_amp = segmentation(norm_rcp_amp[align_y1:align_y2, align_x1:align_x2])

_, shift = align_data(
  data=seg_rcp_amp,
  reference=seg_lcp_amp,
  normalisation="phase"
)

aligned_rcp_amp = fft_shift_data(data=norm_rcp_amp, shift=shift)
aligned_rcp_phase = fft_shift_data(data=norm_rcp_phase, shift=shift)

## XMCD Calculation

In [ ]:
xmcd_amp = np.log(norm_lcp_amp) - np.log(aligned_rcp_amp)
xmcd_phase = norm_lcp_phase - aligned_rcp_phase

fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(16,8))

im1 = ax1.imshow(xmcd_amp[crop:-crop, crop:-crop], cmap="bone", vmin=-0.1, vmax=0.1)
plt.colorbar(im1, ax=ax1)
ax1.set_title("$A_{XMCD}$")

im2 = ax2.imshow(xmcd_phase[crop:-crop, crop:-crop], cmap="bone", vmin=-0.1, vmax=0.1)
plt.colorbar(im2, ax=ax2)
ax2.set_title("$\\Phi_{XMCD}$")

fig.savefig("/tmp/xmcd_plot.png")
plt.show()

## Export Data to h5

In [ ]:
outpath = data_dir + output_dir
Path(outpath).mkdir(parents=True, exist_ok=True)

In [ ]:
with h5py.File(outpath + f"/xmcd_{scan_number_lcp}-{scan_number_rcp}.h5", "w") as s:
    s.create_dataset("lcp_amplitude", data=lcp_amp)
    s.create_dataset("lcp_phase", data=lcp_phase)
    s.create_dataset("rcp_amplitude", data=rcp_amp)
    s.create_dataset("rcp_phase", data=rcp_phase)
    s.create_dataset("xmcd_amplitude", data=xmcd_amp)
    s.create_dataset("xmcd_phase", data=xmcd_phase)
    s.create_dataset("crop", data=crop)
    s.create_dataset("align_x1", data=align_x1)
    s.create_dataset("align_x2", data=align_x2)
    s.create_dataset("align_y1", data=align_y1)
    s.create_dataset("align_y2", data=align_y2)
    s.create_dataset("ramp_x1", data=ramp_x1)
    s.create_dataset("ramp_x2", data=ramp_x2)
    s.create_dataset("ramp_y1", data=ramp_y1)
    s.create_dataset("ramp_y2", data=ramp_y2)
    s.create_dataset("norm_x1", data=norm_x1)
    s.create_dataset("norm_x2", data=norm_x2)
    s.create_dataset("norm_y1", data=norm_y1)
    s.create_dataset("norm_y2", data=norm_y2)